# Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/intro/04_analysis.ipynb)

Official API intro to `minilink.analysis`: linearization, modal / frequency tools,
structural properties, equilibria, and discretization.

**Scripts for depth:** `examples/demos/analysis/`


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## Linearize about an operating point

`plant.linearize(x_bar)` returns an `LTISystem` with $A, B, C, D$ at $\bar x$.
Each matrix is a Jacobian, and `plant.jacobian("f", "x")` reads as $\partial f / \partial x$:
the first name is what you differentiate (`"f"` or an output port), the second
the variable (`"x"`, `"u"`, an input port, `"t"`, or `"params"`).


In [ ]:
import numpy as np
from minilink import InvertedPendulum

plant = InvertedPendulum()
x_bar = np.array([0.0, 0.0])  # upright for this model
lin = plant.linearize(x_bar)
print("A =\n", np.round(lin.A(), 4))
print("B =\n", np.round(lin.B(), 4))
print("poles:", np.round(np.linalg.eigvals(lin.A()), 2))

A = plant.jacobian("f", "x", x_bar)                 # exact (JAX) when the plant traces
A_fd = plant.jacobian("f", "x", x_bar, method="fd")  # central finite differences
print("finite differences close to exact:", np.allclose(A, A_fd, atol=1e-4))
print("df/dparams:", {k: np.round(v, 3) for k, v in plant.jacobian("f", "params", x_bar).items()})


## Structural properties and modal analysis

Controllability / observability and modal tools operate on linearized models.


In [ ]:
from minilink import controllability, observability

ctrl = controllability(lin)
obs = observability(lin)
print("controllable:", ctrl.is_full_rank, f"(rank {ctrl.rank}/{ctrl.n})")
print("observable:  ", obs.is_full_rank, f"(rank {obs.rank}/{obs.n})")

# Modal analysis on the nonlinear plant about the same operating point
plant.modal_analysis(x_bar, mode="all")


## Frequency response, root locus, step response

The control plots read like the MATLAB ones: `plot_bode` (with gain and phase
margins), `plot_pzmap`, `plot_root_locus`, `plot_nyquist`, `plot_step_response`.
Every one takes the same operating point and channel keywords (`of=` / `wrt=`)
and renders with `backend="matplotlib"` or `backend="plotly"`; the data behind
each plot is one call away (`bode`, `pzmap`, `root_locus`, `nyquist`,
`margins`, `step_response`).


In [ ]:
w, mag, phase = plant.bode(x_bar)
print("frequency samples:", len(w), "magnitude shape:", np.shape(mag))

G = plant.transfer_function(x_bar)
print(G.name, " num:", np.round(G.numerator, 3), " den:", np.round(G.denominator, 3))

plant.plot_pzmap(x_bar)
plant.plot_root_locus(x_bar)              # u = -K theta: the poles meet and go imaginary
plant.plot_bode(x_bar, backend="plotly")  # interactive: hover for w, |G|, phase
print("Demos: examples/demos/analysis/analysis_frequency.py, analysis_root_locus.py")


## Equilibria and discretization

Find steady states and map continuous LTI models to discrete-time.


In [ ]:
from minilink import discretize

x_eq = plant.find_equilibrium([0.3, 0.0])
print("equilibrium near [0.3, 0]:", np.round(x_eq, 4))

disc = discretize(plant, dt=0.05)
print("discrete A = d(step)/dx:\n", np.round(disc.jacobian("step", "x", x_bar), 4))
print("Demos: examples/demos/analysis/analysis_equilibrium.py, analysis_discretize.py")
